In [ ]:
import os

os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_DEFAULT_REGION"] = "ru-central1"
os.environ["AWS_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
print("✓ Переменные окружения установлены")

✓ Переменные окружения установлены


In [2]:
# ============================================================================
# GOOGLE COLAB: Yandex Cloud S3 + LitData Streaming Dataset
# ============================================================================
# Копируй и запусти эту ячейку в Google Colab
# Все необходимое в одном месте!

# 1. Установка зависимостей
print("📦 Установка зависимостей...")
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "litdata", "boto3"])
print("✓ Зависимости установлены\n")

# 2. Импорты
import os
import torch
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import boto3
from pathlib import Path
from typing import Optional
import numpy as np

# 3. КОНФИГУРАЦИЯ - ОТРЕДАКТИРУЙ ЭТИ ЗНАЧЕНИЯ!
# ============================================================================
# Способ 1: Прямое указание (не рекомендуется в Colab, используй переменные)
# S3_ACCESS_KEY = "your-access-key"
# S3_SECRET_KEY = "your-secret-key"
# S3_BUCKET = "your-bucket-name"

# Способ 2: Переменные окружения (безопаснее!)
# В Colab используй: from google.colab import userdata
# S3_ACCESS_KEY = userdata.get('YANDEX_S3_ACCESS_KEY')
# S3_SECRET_KEY = userdata.get('YANDEX_S3_SECRET_KEY')
# S3_BUCKET = userdata.get('YANDEX_S3_BUCKET_NAME')

# Для локального тестирования:

# Путь в S3 где находятся твои данные (в формате LitData)
S3_DATA_PREFIX = "yolo_preprocessed/train"  # ← ИЗМЕНИ НА СВОЙ ПУТЬ

# Локальный кеш для данных
CACHE_DIR = "./litdata_cache"  # В Colab можно использовать /tmp


📦 Установка зависимостей...
✓ Зависимости установлены



In [3]:

# ============================================================================
# 4. S3 КЛИЕНТ (self-contained)
# ============================================================================

class YandexS3Client:
    """Клиент для работы с Yandex Cloud S3"""
    
    def __init__(self, access_key, secret_key, bucket_name, endpoint, region):
        self.bucket_name = bucket_name
        self.s3_client = boto3.client(
            "s3",
            # endpoint_url=endpoint,
            # region_name=region,
            # aws_access_key_id=access_key,
            # aws_secret_access_key=secret_key,
        )
    
    def list_objects(self, prefix=""):
        """Список объектов в S3"""
        try:
            response = self.s3_client.list_objects_v2(
                Bucket=self.bucket_name,
                Prefix=prefix,
            )
            return [obj["Key"] for obj in response.get("Contents", [])]
        except Exception as e:
            print(f"✗ Ошибка при получении списка: {e}")
            return []
    
    def get_bucket_size(self, prefix=""):
        """Получить размер данных"""
        total_size = 0
        try:
            paginator = self.s3_client.get_paginator("list_objects_v2")
            pages = paginator.paginate(Bucket=self.bucket_name, Prefix=prefix)
            for page in pages:
                total_size += sum(obj["Size"] for obj in page.get("Contents", []))
            return total_size
        except Exception:
            return 0



In [4]:
import boto3
from typing import Dict


def build_label2idx_s3(bucket: str, prefix: str) -> Dict[str, int]:
    """
    bucket: "my-bucket"
    prefix: "dataset/train/"
    """

    s3 = boto3.client("s3")

    class_names = set()

    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]  # например: dataset/train/cat/img1.jpg

            parts = key.split("/")
            if len(parts) >= 2:
                class_name = parts[-2]  # папка перед файлом
                class_names.add(class_name)

    class_names = sorted(class_names)
    return {cls: i for i, cls in enumerate(class_names)}

In [7]:
label2idx = build_label2idx_s3(
    bucket='sneakers-hse-images-test',
    prefix='yolo_preprocessed/search-dataset-images/'
)



In [ ]:
import pickle
label2idx

with open(".")

{'Kari Кеды': 0,
 'Kari Кроссовки': 1,
 'Maison David Кеды': 2,
 'Maison David Кроссовки': 3,
 'Nike Кеды Dunk Low Retro': 4,
 'Nike Кеды FIELD GENERAL': 5,
 'Nike Кеды NIKE AIR FORCE 1': 6,
 'Nike Кроссовки AIR MAX 90': 7,
 'Nike Кроссовки AIR PEGASUS 2005': 8,
 'Nike Кроссовки Nike Zoom Vomero 5': 9,
 'PUMA Кеды CA Pro Classic II': 10,
 'PUMA Кеды Palermo': 11,
 'PUMA Кроссовки Darter Pro': 12,
 'PUMA Кроссовки Hypnotic LS': 13,
 'PUMA Кроссовки Puma Morphic': 14,
 'PUMA Кроссовки RS-X Efekt PRM': 15,
 'Reebok Кеды CLUB C 85': 16,
 'Reebok Кеды CLUB C GROUNDS UK': 17,
 'Reebok Кроссовки CLASSIC LEATHER': 18,
 'Reebok Кроссовки CLASSIC NYLON': 19,
 'Reebok Кроссовки RBK PREMIER TRINITY': 20,
 'Tendance Кеды': 21,
 'Tendance Кроссовки': 22,
 'Under Armour Кроссовки UA CHARGED EDGE': 23,
 'Under Armour Кроссовки UA CHARGED SPEED SWIFT': 24,
 'Under Armour Кроссовки UA Charged Rogue 5': 25,
 'Under Armour Кроссовки UA Charged Surge 4': 26,
 'Under Armour Кроссовки UA Phantom 4': 27,
 'Va

In [13]:
from litdata import StreamingDataLoader, StreamingDataset

In [ ]:
class LitDataImageDataset(Dataset):
    def __init__(self, input_dir, label2idx, transform=None, train=True):
        self.ds = StreamingDataset(input_dir, shuffle=train, drop_last=train)
        self.transform = transform
        self.label2idx = label2idx
    
    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        image = item["image"]
        label = item["label"].split("/")[-1]  # извлекаем имя класса из пути
        label = self.label2idx[label]

        # Это надо для albumentations
        if image.ndim == 3 and image.shape[0] in (1, 3):
            image = image.permute(1, 2, 0)
        image = image.cpu().numpy()

        image = self.transform(image=image)["image"]

        return image, label



In [23]:
is_transforms = True

In [24]:

# Проверка учетных данных
    # Инициализация S3 клиента
s3_client = YandexS3Client(
    access_key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    bucket_name=os.getenv("AWS_BUCKET_NAME"),
    endpoint=os.getenv("AWS_ENDPOINT_URL"),
    region=os.getenv("AWS_DEFAULT_REGION"),
)
print("✓ S3 клиент инициализирован\n")

# Проверка доступности данных в S3
print(f"📊 Проверка данных в S3...")
objects = s3_client.list_objects(prefix=S3_DATA_PREFIX)
print(f"   Найдено объектов: {len(objects)}")

if objects:
    print("   Примеры файлов:")
    for obj in objects[:5]:
        print(f"     - {obj}")
    
    size_gb = s3_client.get_bucket_size(S3_DATA_PREFIX) / (1024**3)
    print(f"   Общий размер: {size_gb:.2f} GB\n")

# ========================================================================
# 7. СОЗДАНИЕ DATASET И DATALOADER
# ========================================================================

print("🎯 Создание Dataset...")

# Определи трансформации
transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),  # Только для train
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
    ToTensorV2(),
], is_check_shapes=False)

# Путь к данным в S3
s3_path = "s3://sneakers-hse-images-test/yolo_preprocessed/train"

# Создание dataset
if is_transforms:
    dataset = LitDataImageDataset(
        input_dir=s3_path,
        label2idx=label2idx,
        transform=transforms,
        # cache_dir=CACHE_DIR,
    )
else:
    dataset = StreamingDataset(
        input_dir=s3_path,
        cache_dir=CACHE_DIR
    )

print(f"✓ Dataset готов: {len(dataset)} samples\n")

# Параметры для DataLoader
BATCH_SIZE = 32
NUM_WORKERS = 4

# Создание dataloader
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    #shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"✓ DataLoader готов:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   NumWorkers: {NUM_WORKERS}\n")


print(f"✓ Dataset готов: {len(dataset)} samples\n")

# Параметры для DataLoader
BATCH_SIZE = 8
NUM_WORKERS = 4

# Создание dataloader
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"✓ DataLoader готов:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   NumWorkers: {NUM_WORKERS}\n")


✓ S3 клиент инициализирован

📊 Проверка данных в S3...
✗ Ошибка при получении списка: expected string or bytes-like object, got 'NoneType'
   Найдено объектов: 0
🎯 Создание Dataset...
✓ Dataset готов: 11265 samples

✓ DataLoader готов:
   Batch size: 32
   NumWorkers: 4

✓ Dataset готов: 11265 samples

✓ DataLoader готов:
   Batch size: 8
   NumWorkers: 4



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [25]:
for i in range(8):
    print(i)
    print(dataset[i])

0
(tensor([[[1.9920, 1.9920, 1.9920,  ..., 1.9920, 1.9920, 1.9920],
         [1.9920, 1.9920, 1.9920,  ..., 1.9920, 1.9920, 1.9920],
         [1.9920, 1.9920, 1.9920,  ..., 1.9920, 1.9920, 1.9920],
         ...,
         [1.9920, 1.9920, 1.9920,  ..., 1.9920, 1.9920, 1.9920],
         [1.9920, 1.9920, 1.9920,  ..., 1.9920, 1.9920, 1.9920],
         [1.9920, 1.9920, 1.9920,  ..., 1.9920, 1.9920, 1.9920]],

        [[2.1660, 2.1660, 2.1660,  ..., 2.1660, 2.1660, 2.1660],
         [2.1660, 2.1660, 2.1660,  ..., 2.1660, 2.1660, 2.1660],
         [2.1660, 2.1660, 2.1660,  ..., 2.1660, 2.1660, 2.1660],
         ...,
         [2.1660, 2.1660, 2.1660,  ..., 2.1660, 2.1660, 2.1660],
         [2.1660, 2.1660, 2.1660,  ..., 2.1660, 2.1660, 2.1660],
         [2.1660, 2.1660, 2.1660,  ..., 2.1660, 2.1660, 2.1660]],

        [[2.3786, 2.3786, 2.3786,  ..., 2.3786, 2.3786, 2.3786],
         [2.3786, 2.3786, 2.3786,  ..., 2.3786, 2.3786, 2.3786],
         [2.3786, 2.3786, 2.3786,  ..., 2.3786, 2.3786,

In [143]:
dataset[0][1]

'yolo_preprocessed/search-dataset-images/Kari Кеды'

In [27]:

# ========================================================================
# 8. ТЕСТИРОВАНИЕ (загрузи первый батч)
# ========================================================================

print("🧪 Загрузка первого батча...")
for images, labels in dataloader:
    print(f"✓ Успешно!")
    print(f"   Images shape: {images.shape}")
    print(f"   Labels shape: {labels.shape}")
    print(f"   Labels values: {labels[:10]}")
    print(f"   Images dtype: {images.dtype}")
    print(f"   Images range: [{images.min():.3f}, {images.max():.3f}]")
    break


🧪 Загрузка первого батча...
✓ Успешно!
   Images shape: torch.Size([8, 3, 224, 224])
   Labels shape: torch.Size([8])
   Labels values: tensor([0, 0, 0, 0, 0, 0, 0, 0])
   Images dtype: torch.float32
   Images range: [-2.118, 2.640]


In [28]:
torch.tensor(labels)

/tmp/ipykernel_43495/2767672595.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(labels)


tensor([0, 0, 0, 0, 0, 0, 0, 0])

In [29]:
# Получить первый sample
image, label = dataset[3]
print(f"Image shape: {image.shape}, Label: {label}")

Image shape: torch.Size([3, 224, 224]), Label: 0


In [ ]:
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


In [ ]:
for image, label in dataloader:
    print(f"Image shape: {image.shape}, Label: {label}")
    break

AttributeError: Caught AttributeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 292, in _worker_loop
    fetcher = _DatasetKind.create_fetcher(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 89, in create_fetcher
    return _utils.fetch._IterableDatasetFetcher(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 24, in __init__
    self.dataset_iter = iter(dataset)
                        ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litdata/streaming/dataset.py", line 363, in __iter__
    self.cache = self._create_cache(worker_env=self.worker_env)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litdata/streaming/dataset.p